# Máquinas de vetores de suporte (SVM)
## Monitoria 4: classificação e regressão com a "rua mais larga"

**Pré-requisito:** Monitorias 2 e 3 (scikit-learn, pipelines, validação cruzada, *overfitting* e regularização) · **Ambiente:** Google Colab · **Duração:** 2h30–3h 

**Base:** capítulo 5 (*Support Vector Machines*) do livro *Hands-On Machine Learning*, de Aurélien Géron ([notebooks no GitHub](https://github.com/ageron/handson-ml3)).

### Como estudar este notebook

Esta monitoria foi preparada para ser feita **por conta própria**. Algumas dicas:

- **Leia o texto antes de cada célula**, execute a célula (`Shift + Enter`) e compare o resultado com o que o texto diz.
- **Execute as células em ordem.** Várias seções reaproveitam variáveis e funções criadas antes. Se algo der erro estranho, use **Ambiente de execução → Executar anteriores**.
- Células que começam com `# Código do gráfico` só desenham figuras. **Não é preciso entender cada linha**: concentre-se no que a figura mostra.
- Os blocos **▶ Pergunta** abrem com um clique e trazem a resposta. **Tente responder antes de abrir.**
- Se travar, use a IA (seção 7) para pedir uma explicação diferente.

### Objetivo
Entender a ideia central das SVM (separar as classes pela **"rua" mais larga possível**), saber quando usar uma SVM linear ou com *kernel*, ajustar os hiperparâmetros `C`, `gamma` e `epsilon` e usar SVM também para **regressão**.

**Pergunta-guia:** quando muitas retas separam duas classes, qual delas escolher? E o que fazer quando nenhuma reta separa?

### Roteiro

| Bloco | Conteúdo | Tempo |
|---|---|---|
| 1 | Classificação linear: margem larga, vetores de suporte e o hiperparâmetro `C` | 35 min |
| 2 | Classificação não linear: variáveis polinomiais, truque do *kernel* e *kernel* RBF | 40 min |
| — | Intervalo | 10 min |
| 3 | Regressão com SVM | 15 min |
| 4 | Por dentro da SVM: função de decisão e *hinge loss* (*opcional*) | 15 min |
| 5 | SVM "do zero" com gradiente descendente (*opcional*) | casa |
| 6 | Exercícios resolvidos do capítulo (9, 10 e 11) | 30 min |
| 7 | IA e fechamento | 10 min |

As seções marcadas como *opcional* aprofundam a matemática e podem ser deixadas para depois.

### Vocabulário da aula

Volte a esta tabela sempre que um termo parecer estranho.

| Termo | O que significa |
|---|---|
| **Fronteira de decisão** | A linha (ou curva) que separa o que o modelo classifica como uma classe ou outra. |
| **Margem** ("rua") | A faixa livre em torno da fronteira. A SVM procura a rua **mais larga** que separa as classes. |
| **Vetores de suporte** | As observações que ficam **na borda ou dentro da rua**. São elas que definem a fronteira. |
| **Margem rígida** (*hard margin*) | Nenhuma observação pode ficar dentro da rua ou do lado errado. |
| **Margem suave** (*soft margin*) | Algumas observações podem invadir a rua; é um equilíbrio entre rua larga e poucas invasões. |
| **`C`** | Hiperparâmetro de regularização. `C` **pequeno** → rua larga, mais invasões, **mais** regularização. `C` **grande** → rua estreita, menos invasões, **menos** regularização. |
| ***Kernel*** | Uma função de "similaridade" que permite à SVM traçar fronteiras curvas sem criar novas colunas de fato. |
| **`gamma`** | No *kernel* RBF, controla o alcance da influência de cada observação. `gamma` **grande** → fronteira mais irregular. |
| **`epsilon`** ($\epsilon$) | Na regressão com SVM, a largura da rua dentro da qual os erros são ignorados. |

## 0. Preparação

Todas as bibliotecas já estão instaladas no Colab. A célula abaixo importa o que vamos usar e aumenta o tamanho padrão da fonte dos gráficos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn

np.random.seed(42)

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

print("scikit-learn:", sklearn.__version__)

## 1. Classificação linear com SVM

### 1.1 A ideia central: a rua mais larga

Vamos voltar ao conjunto **Iris** da Monitoria 3 (150 flores de 3 espécies). Por enquanto, usamos só duas espécies, *setosa* e *versicolor*, e duas variáveis: **comprimento** e **largura da pétala**. Essas duas espécies são **linearmente separáveis**: dá para traçar uma reta com todas as *setosa* de um lado e todas as *versicolor* do outro.

O problema é que **existem infinitas retas** que fazem isso. Qual escolher?

A resposta da SVM: escolha a reta que deixa a **maior distância possível** até as observações mais próximas de cada classe. Imagine que você precisa construir uma **rua** entre dois bairros sem derrubar nenhuma casa: a SVM procura a rua **mais larga** possível. A fronteira de decisão é o canteiro central dessa rua.

Por que isso é bom? Uma rua larga dá **folga**: se uma flor nova vier um pouco diferente das que vimos no treino, ela ainda tende a cair do lado certo.

Primeiro, carregamos os dados e treinamos a SVM. A classe `SVC` (*Support Vector Classifier*) com `kernel="linear"` traça uma fronteira reta. O valor enorme de `C` força uma **margem rígida** (explicamos `C` na seção 1.3).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC

iris = load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris.target.values  # 0 = setosa, 1 = versicolor, 2 = virginica

setosa_ou_versicolor = (y == 0) | (y == 1)  # descartamos a virginica por enquanto
X = X[setosa_ou_versicolor]
y = y[setosa_ou_versicolor]

svm_clf = SVC(kernel="linear", C=1e100)  # C enorme = margem rígida
svm_clf.fit(X, y)

print("Número de flores no treino:", len(X))
print("Vetores de suporte (comprimento, largura):")
print(svm_clf.support_vectors_)

Das 100 flores, só **duas** são vetores de suporte: são as flores que tocam a borda da rua. Agora vamos desenhar.

A célula a seguir cria duas funções de desenho que usaremos na aula toda:

- `plotar_rua(w, b, ...)` desenha a fronteira (linha cheia), as bordas da rua (linhas tracejadas) e destaca os vetores de suporte (círculos cinza).
- `plotar_fronteira_svc(modelo, ...)` faz o mesmo a partir de um modelo `SVC` já treinado.

In [ ]:
# Código do gráfico: funções auxiliares para desenhar a "rua" de uma SVM linear

def plotar_rua(w, b, xmin, xmax, vetores_suporte=None):
    # Na fronteira, w0*x0 + w1*x1 + b = 0  =>  x1 = -w0/w1 * x0 - b/w1
    x0 = np.linspace(xmin, xmax, 200)
    fronteira = -w[0] / w[1] * x0 - b / w[1]
    margem = 1 / w[1]  # as bordas da rua ficam onde w0*x0 + w1*x1 + b = +1 ou -1

    plt.plot(x0, fronteira, "k-", linewidth=2, zorder=-2)
    plt.plot(x0, fronteira + margem, "k--", linewidth=2, zorder=-2)
    plt.plot(x0, fronteira - margem, "k--", linewidth=2, zorder=-2)
    if vetores_suporte is not None:
        plt.scatter(vetores_suporte[:, 0], vetores_suporte[:, 1], s=180,
                    facecolors="#AAA", zorder=-1)


def plotar_fronteira_svc(modelo, xmin, xmax):
    plotar_rua(modelo.coef_[0], modelo.intercept_[0], xmin, xmax,
               modelo.support_vectors_)

In [ ]:
# Código do gráfico: retas quaisquer x reta da SVM

x0 = np.linspace(0, 5.5, 200)
reta_1 = 5 * x0 - 20     # nem separa as classes
reta_2 = x0 - 1.8        # separa, mas passa muito perto das flores
reta_3 = 0.1 * x0 + 0.5  # separa, mas passa muito perto das flores

fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)

plt.sca(axes[0])
plt.plot(x0, reta_1, "g--", linewidth=2)
plt.plot(x0, reta_2, "m-", linewidth=2)
plt.plot(x0, reta_3, "r-", linewidth=2)
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "bs", label="Iris versicolor")
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "yo", label="Iris setosa")
plt.xlabel("Comprimento da pétala")
plt.ylabel("Largura da pétala")
plt.legend(loc="upper left", fontsize=11)
plt.axis([0, 5.5, 0, 2])
plt.gca().set_aspect("equal")
plt.title("Retas quaisquer")
plt.grid(alpha=0.25)

plt.sca(axes[1])
plotar_fronteira_svc(svm_clf, 0, 5.5)
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "bs")
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "yo")
plt.xlabel("Comprimento da pétala")
plt.axis([0, 5.5, 0, 2])
plt.gca().set_aspect("equal")
plt.title("SVM: a rua mais larga")
plt.grid(alpha=0.25)

plt.show()

**O que observar**
- **À esquerda:** a reta verde tracejada nem separa as classes. As retas roxa e vermelha separam, mas passam **colada** nas flores. Uma flor nova um pouco diferente cairia do lado errado.
- **À direita:** a linha cheia é a fronteira da SVM, e as tracejadas são as bordas da rua. A fronteira fica o **mais longe possível** das flores mais próximas de cada espécie.
- Os círculos cinza são os **vetores de suporte**: as flores que "seguram" as bordas da rua.

<details><summary><b>▶ Pergunta:</b> por que o nome "vetores de suporte"? E se eu adicionar mais flores longe da rua, a fronteira muda?</summary>
<br>
"Vetor" é só o nome matemático de uma observação (uma lista de números, como <code>[comprimento, largura]</code>). Eles são "de suporte" porque <b>sustentam</b> a rua, como colunas: a fronteira depende <b>apenas</b> deles.<br><br>
Por isso, adicionar, remover ou mover flores que estão <b>fora da rua</b> não muda nada na fronteira. Só mudam o modelo as observações que caem na borda ou dentro da rua. Essa é uma diferença importante em relação à regressão logística, em que todas as observações influenciam o resultado.
</details>

### 1.2 SVM é sensível à escala das variáveis

A largura da rua é medida em **distância**. Se uma variável vai de 0 a 90 e a outra de 0 a 6, a primeira domina o cálculo de distância, e a SVM praticamente **ignora** a variável de escala pequena.

No exemplo abaixo, com só quatro pontos, $x_1$ varia de 20 a 80 e $x_0$ de 1 a 5. À esquerda, sem padronização, a rua mais larga fica quase **horizontal** (só usa $x_1$). À direita, depois do `StandardScaler` (Monitoria 2), as duas variáveis contam e a rua fica bem melhor.

In [ ]:
# Código do gráfico: efeito da escala das variáveis

from sklearn.preprocessing import StandardScaler

Xs = np.array([[1, 50], [5, 20], [3, 80], [5, 60]]).astype(np.float64)
ys = np.array([0, 0, 1, 1])
svm_sem_escala = SVC(kernel="linear", C=100).fit(Xs, ys)

scaler = StandardScaler()
X_padronizado = scaler.fit_transform(Xs)
svm_com_escala = SVC(kernel="linear", C=100).fit(X_padronizado, ys)

plt.figure(figsize=(9, 2.7))
plt.subplot(121)
plt.plot(Xs[:, 0][ys == 1], Xs[:, 1][ys == 1], "bo")
plt.plot(Xs[:, 0][ys == 0], Xs[:, 1][ys == 0], "ms")
plotar_fronteira_svc(svm_sem_escala, 0, 6)
plt.xlabel("$x_0$")
plt.ylabel("$x_1$    ", rotation=0)
plt.title("Sem padronização")
plt.axis([0, 6, 0, 90])
plt.grid(alpha=0.25)

plt.subplot(122)
plt.plot(X_padronizado[:, 0][ys == 1], X_padronizado[:, 1][ys == 1], "bo")
plt.plot(X_padronizado[:, 0][ys == 0], X_padronizado[:, 1][ys == 0], "ms")
plotar_fronteira_svc(svm_com_escala, -2, 2)
plt.xlabel("$x'_0$")
plt.ylabel("$x'_1$  ", rotation=0)
plt.title("Com padronização")
plt.axis([-2, 2, -2, 2])
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

**Regra prática:** com SVM, **sempre padronize as variáveis**. O jeito mais seguro é colocar o `StandardScaler` dentro de um `Pipeline` (Monitoria 2), como faremos daqui em diante.

<details><summary><b>▶ Pergunta:</b> num problema de negócios, onde isso apareceria?</summary>
<br>
Em quase todos. Imagine classificar clientes com <b>renda mensal</b> (em reais, de 1.000 a 50.000) e <b>número de compras no ano</b> (de 0 a 30). Sem padronizar, a SVM mede distâncias quase só pela renda, e o número de compras é praticamente ignorado, mesmo que seja a variável mais informativa.
</details>

### 1.3 Margem rígida x margem suave e o hiperparâmetro `C`

Até aqui exigimos que **todas** as flores ficassem fora da rua e do lado certo. Isso é a **margem rígida**. Ela tem dois problemas:

1. Só funciona se os dados forem **linearmente separáveis**.
2. É muito **sensível a valores atípicos** (*outliers*).

Veja o que acontece quando acrescentamos uma única flor estranha:

In [ ]:
# Código do gráfico: margem rígida com valores atípicos

X_atipicos = np.array([[3.4, 1.3], [3.2, 0.8]])
y_atipicos = np.array([0, 0])  # duas "setosas" estranhas
Xo1 = np.concatenate([X, X_atipicos[:1]], axis=0)
yo1 = np.concatenate([y, y_atipicos[:1]], axis=0)
Xo2 = np.concatenate([X, X_atipicos[1:]], axis=0)
yo2 = np.concatenate([y, y_atipicos[1:]], axis=0)

svm_rigida = SVC(kernel="linear", C=10**9)
svm_rigida.fit(Xo2, yo2)

fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)

plt.sca(axes[0])
plt.plot(Xo1[:, 0][yo1 == 1], Xo1[:, 1][yo1 == 1], "bs")
plt.plot(Xo1[:, 0][yo1 == 0], Xo1[:, 1][yo1 == 0], "yo")
plt.text(0.3, 1.0, "Impossível!", color="red", fontsize=18)
plt.xlabel("Comprimento da pétala")
plt.ylabel("Largura da pétala")
plt.annotate("Atípico", xy=(X_atipicos[0][0], X_atipicos[0][1]), xytext=(2.5, 1.7),
             ha="center", arrowprops=dict(facecolor="black", shrink=0.1))
plt.axis([0, 5.5, 0, 2])
plt.grid(alpha=0.25)

plt.sca(axes[1])
plt.plot(Xo2[:, 0][yo2 == 1], Xo2[:, 1][yo2 == 1], "bs")
plt.plot(Xo2[:, 0][yo2 == 0], Xo2[:, 1][yo2 == 0], "yo")
plotar_fronteira_svc(svm_rigida, 0, 5.5)
plt.xlabel("Comprimento da pétala")
plt.annotate("Atípico", xy=(X_atipicos[1][0], X_atipicos[1][1]), xytext=(3.2, 0.08),
             ha="center", arrowprops=dict(facecolor="black", shrink=0.1))
plt.axis([0, 5.5, 0, 2])
plt.grid(alpha=0.25)

plt.show()

**O que observar**
- **À esquerda:** a flor atípica está no meio das *versicolor*. Nenhuma reta separa as classes, então a margem rígida é **impossível**.
- **À direita:** outra flor atípica fica perto das *versicolor*. A margem rígida ainda existe, mas a rua ficou **muito mais estreita** e a fronteira mudou bastante por causa de **uma única** flor. Provavelmente esse modelo vai generalizar mal.

A solução é a **margem suave** (*soft margin classification*): aceitar que algumas observações invadam a rua (as **violações de margem**) em troca de uma rua mais larga. O equilíbrio é controlado pelo hiperparâmetro **`C`**:

| `C` | Rua | Violações de margem | Regularização |
|---|---|---|---|
| **pequeno** | larga | mais | **mais** forte (menos risco de *overfitting*) |
| **grande** | estreita | menos | **mais** fraca (mais risco de *overfitting*) |

Agora usamos **as três espécies** do Iris e treinamos um classificador binário: *virginica* ou não. A classe `LinearSVC` é uma implementação **especializada em SVM linear**, mais rápida que `SVC(kernel="linear")`. Note o `StandardScaler` no pipeline.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 2).values  # True se for Iris virginica

svm_clf = make_pipeline(StandardScaler(),
                        LinearSVC(C=1, random_state=42))
svm_clf.fit(X, y)

Agora fazemos previsões para duas flores novas: uma com pétala de 5,5 × 1,7 cm e outra de 5,0 × 1,5 cm.

In [ ]:
X_novas = [[5.5, 1.7], [5.0, 1.5]]

print("É virginica?          ", svm_clf.predict(X_novas))
print("Pontuação (decisão):  ", svm_clf.decision_function(X_novas).round(2))

O `decision_function` dá a **pontuação** de cada flor, uma medida (com sinal) da distância até a fronteira:

- **pontuação positiva** → lado da classe positiva (*virginica*);
- **pontuação negativa** → lado da classe negativa;
- quanto **mais longe de zero**, mais longe da fronteira, ou seja, maior a "confiança".

Lembre da Monitoria 2: era a mesma pontuação que usávamos para escolher o limite (*threshold*) de decisão.

<details><summary><b>▶ Pergunta:</b> a pontuação é uma probabilidade? Posso dizer que a primeira flor tem 66% de chance de ser virginica?</summary>
<br>
<b>Não.</b> A pontuação da SVM é uma distância, não uma probabilidade: ela pode ser maior que 1 ou negativa, e não soma 1 entre as classes.<br><br>
Se você precisar de probabilidades (por exemplo, para calcular um valor esperado de lucro), há duas saídas: usar <code>SVC(probability=True)</code>, que ao final do treino ajusta uma regressão logística sobre as pontuações usando validação cruzada (o treino fica mais lento), e então chamar <code>predict_proba()</code>; ou usar diretamente um modelo que já estima probabilidades, como a regressão logística.
</details>

Agora vejamos o efeito de `C`. Treinamos dois modelos, com `C = 1` e `C = 100`. A célula é longa só por causa do desenho: como o modelo foi treinado com os dados padronizados, convertemos os parâmetros de volta para centímetros antes de desenhar.

In [ ]:
# Código do gráfico: efeito de C na largura da rua

def parametros_na_escala_original(pipeline):
    # Converte w e b do modelo treinado em dados padronizados para a escala original
    escala, svm = pipeline[0], pipeline[-1]
    w = svm.coef_[0] / escala.scale_
    b = svm.intercept_[0] - (svm.coef_[0] * escala.mean_ / escala.scale_).sum()
    return w, b

def vetores_de_suporte(X, y, w, b):
    # O LinearSVC não guarda os vetores de suporte: são as flores na borda ou dentro da rua
    t = y * 2 - 1  # +1 para virginica, -1 para as demais
    return X[t * (X @ w + b) < 1]

modelos = {}
for C in (1, 100):
    modelo = make_pipeline(StandardScaler(),
                           LinearSVC(C=C, max_iter=10_000, random_state=42))
    modelos[C] = modelo.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)
for ax, C in zip(axes, modelos):
    plt.sca(ax)
    w, b = parametros_na_escala_original(modelos[C])
    plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^", label="Iris virginica")
    plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs", label="Iris versicolor")
    plotar_rua(w, b, 4, 5.99, vetores_de_suporte(X, y, w, b))
    plt.xlabel("Comprimento da pétala")
    plt.title(f"$C = {C}$")
    plt.axis([4, 5.9, 0.8, 2.8])
    plt.grid(alpha=0.25)
    print(f"C = {C:>3}: {len(vetores_de_suporte(X, y, w, b))} flores na borda ou dentro da rua")

plt.sca(axes[0])
plt.ylabel("Largura da pétala")
plt.legend(loc="upper left", fontsize=11)
plt.show()

**Interprete**

<details><summary><b>▶ Pergunta 1:</b> qual dos dois modelos tem a rua mais larga? Qual tem mais violações de margem?</summary>
<br>
O modelo com <code>C = 1</code> (esquerda) tem a rua bem mais larga e, por isso, muitas flores ficam dentro dela (mais violações). O modelo com <code>C = 100</code> (direita) tem a rua estreita e poucas violações.
</details>

<details><summary><b>▶ Pergunta 2:</b> qual dos dois você esperaria que generalizasse melhor para flores novas?</summary>
<br>
Provavelmente o de <code>C = 1</code>. A maioria das violações de margem dele está do <b>lado certo da fronteira</b>: essas flores estão dentro da rua, mas ainda são classificadas corretamente. Uma rua larga é menos sensível a detalhes específicos da amostra de treino. Para ter certeza, compararíamos os dois modelos por <b>validação cruzada</b> (Monitoria 2).
</details>

<details><summary><b>▶ Pergunta 3:</b> meu modelo SVM está com <i>overfitting</i>. O que faço com <code>C</code>?</summary>
<br>
<b>Diminua <code>C</code></b>. Isso aumenta a regularização: a rua fica mais larga e o modelo passa a ignorar detalhes do treino. Se estiver com <i>underfitting</i>, faça o contrário e aumente <code>C</code>.
</details>

<details><summary><b>▶ Pergunta 4:</b> este <code>C</code> é o mesmo da <code>LogisticRegression</code> da Monitoria 3?</summary>
<br>
É a mesma ideia: nos dois casos <code>C</code> é o <b>inverso</b> da força da regularização (quanto maior <code>C</code>, menos regularização). É o contrário do <code>alpha</code> da Ridge e da Lasso, em que um valor maior significa <b>mais</b> regularização.
</details>

## 2. Classificação não linear com SVM

Muitas vezes **nenhuma reta** separa as classes. Uma saída é a mesma da regressão polinomial da Monitoria 3: **criar novas variáveis** a partir das que temos.

### 2.1 A ideia: ganhar uma dimensão

À esquerda, temos uma única variável $x_1$. As classes (quadrados azuis e triângulos verdes) estão "misturadas" na reta: os verdes no meio e os azuis nas pontas. Nenhum ponto de corte separa os dois grupos.

À direita, criamos uma segunda variável, $x_2 = x_1^2$. Agora cada ponto tem duas coordenadas, $(x_1, x_1^2)$, e **uma reta horizontal** (tracejada vermelha) separa perfeitamente as classes.

In [ ]:
# Código do gráfico: acrescentar x2 = x1² torna os dados separáveis

X1D = np.linspace(-4, 4, 9).reshape(-1, 1)
X2D = np.c_[X1D, X1D ** 2]
y1d = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

plt.figure(figsize=(10, 3))

plt.subplot(121)
plt.grid(alpha=0.25)
plt.axhline(y=0, color="k")
plt.plot(X1D[:, 0][y1d == 0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][y1d == 1], np.zeros(5), "g^")
plt.gca().get_yaxis().set_ticks([])
plt.xlabel("$x_1$")
plt.title("Uma variável: não separável")
plt.axis([-4.5, 4.5, -0.2, 0.2])

plt.subplot(122)
plt.grid(alpha=0.25)
plt.axhline(y=0, color="k")
plt.axvline(x=0, color="k")
plt.plot(X2D[:, 0][y1d == 0], X2D[:, 1][y1d == 0], "bs")
plt.plot(X2D[:, 0][y1d == 1], X2D[:, 1][y1d == 1], "g^")
plt.xlabel("$x_1$")
plt.ylabel("$x_2 = x_1^2$")
plt.gca().get_yaxis().set_ticks([0, 4, 8, 12, 16])
plt.plot([-4.5, 4.5], [6.5, 6.5], "r--", linewidth=3)
plt.title("Duas variáveis: separável")
plt.axis([-4.5, 4.5, -1, 17])

plt.tight_layout()
plt.show()

### 2.2 Variáveis polinomiais + SVM linear

Vamos testar a ideia num conjunto clássico de brinquedo, as **luas** (`make_moons`): duas classes em forma de meia-lua, encaixadas uma na outra. Nenhuma reta separa as luas.

Montamos um pipeline com três etapas:
1. `PolynomialFeatures(degree=3)`: cria todas as potências e interações até o grau 3 ($x_1^2$, $x_1 x_2$, $x_2^3$, ...).
2. `StandardScaler()`: padroniza (lembre da seção 1.2).
3. `LinearSVC`: SVM linear, mas agora nas variáveis expandidas.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures

X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

polynomial_svm_clf = make_pipeline(
    PolynomialFeatures(degree=3),
    StandardScaler(),
    LinearSVC(C=10, max_iter=10_000, random_state=42),
)
polynomial_svm_clf.fit(X, y)

print("Variáveis originais:", X.shape[1])
print("Variáveis depois do PolynomialFeatures:",
      polynomial_svm_clf[0].transform(X[:1]).shape[1])

De 2 variáveis passamos a 10. Vamos ver a fronteira resultante. As funções abaixo também serão usadas no resto da seção: `plotar_dados` desenha as luas e `plotar_previsoes` pinta o fundo com a classe prevista em cada ponto do plano.

In [ ]:
# Código do gráfico: funções para desenhar as luas e as regiões previstas

def plotar_dados(X, y, eixos):
    plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs")
    plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^")
    plt.axis(eixos)
    plt.grid(alpha=0.25)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$", rotation=0)


def plotar_previsoes(modelo, eixos):
    # Cria uma grade de 100 x 100 pontos e prevê a classe de cada um
    x0s = np.linspace(eixos[0], eixos[1], 100)
    x1s = np.linspace(eixos[2], eixos[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_grade = np.c_[x0.ravel(), x1.ravel()]
    y_pred = modelo.predict(X_grade).reshape(x0.shape)
    y_decisao = modelo.decision_function(X_grade).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)
    plt.contourf(x0, x1, y_decisao, cmap=plt.cm.brg, alpha=0.1)


plt.figure(figsize=(6, 4))
plotar_previsoes(polynomial_svm_clf, [-1.5, 2.5, -1, 1.5])
plotar_dados(X, y, [-1.5, 2.5, -1, 1.5])
plt.title("LinearSVC com variáveis polinomiais de grau 3")
plt.show()

A fronteira agora é **curva** e acompanha bem o formato das luas, embora o modelo seja uma SVM **linear**: ela é linear nas 10 variáveis criadas, não nas 2 originais. É o mesmo truque da regressão polinomial.

**O problema:** com muitas variáveis originais e grau alto, o número de colunas criadas **explode** (você viu isso na Monitoria 3). O modelo fica lento e pesado. É aqui que entra o *kernel*.

### 2.3 *Kernel* polinomial

O **truque do *kernel*** (*kernel trick*) permite obter **o mesmo resultado** que teríamos ao adicionar muitas variáveis polinomiais, **sem criá-las de fato**. Não há explosão de colunas, porque nada é criado.

Na prática, basta trocar o `LinearSVC` + `PolynomialFeatures` por um `SVC(kernel="poly")`:

- `degree`: o grau do polinômio;
- `coef0`: o peso dos termos de grau alto em relação aos de grau baixo;
- `C`: o mesmo hiperparâmetro de regularização de antes.

In [ ]:
poly_kernel_svm_clf = make_pipeline(StandardScaler(),
                                    SVC(kernel="poly", degree=3, coef0=1, C=5))
poly_kernel_svm_clf.fit(X, y)

In [ ]:
# Código do gráfico: kernel polinomial de grau 3 x grau 10

poly100_kernel_svm_clf = make_pipeline(StandardScaler(),
                                       SVC(kernel="poly", degree=10, coef0=100, C=5))
poly100_kernel_svm_clf.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(10.5, 4), sharey=True)

plt.sca(axes[0])
plotar_previsoes(poly_kernel_svm_clf, [-1.5, 2.45, -1, 1.5])
plotar_dados(X, y, [-1.5, 2.4, -1, 1.5])
plt.title("degree=3, coef0=1, C=5")

plt.sca(axes[1])
plotar_previsoes(poly100_kernel_svm_clf, [-1.5, 2.45, -1, 1.5])
plotar_dados(X, y, [-1.5, 2.4, -1, 1.5])
plt.title("degree=10, coef0=100, C=5")
plt.ylabel("")

plt.show()

**O que observar:** à esquerda, grau 3 dá uma fronteira suave e razoável. À direita, grau 10 dá uma fronteira mais "torcida", que se ajusta mais aos detalhes do treino. Se o modelo tiver *overfitting*, reduza o grau; se tiver *underfitting*, aumente.

<details><summary><b>▶ Pergunta:</b> como o truque do <i>kernel</i> funciona, sem muita matemática?</summary>
<br>
Durante o treino, a SVM (na sua forma chamada "dual") <b>não precisa das variáveis transformadas em si</b>. Ela só precisa saber <b>quão parecidos</b> são os pares de observações, o que se mede pelo produto escalar entre elas.<br><br>
Um <i>kernel</i> é uma função que calcula essa "semelhança" <b>como se</b> as observações estivessem no espaço com todas as variáveis polinomiais, mas usando só as variáveis originais. Exemplo com grau 2: para dois pontos <code>a</code> e <code>b</code>, calcular <code>(a · b)²</code> com 2 variáveis dá exatamente o mesmo número que transformar os dois pontos em <code>(a₁², √2·a₁a₂, a₂²)</code> e depois fazer o produto escalar. O resultado é o mesmo, com muito menos conta.<br><br>
A seção 4 traz um pouco mais da matemática, para quem quiser.
</details>

### 2.4 Variáveis de similaridade

Outra forma de criar variáveis é medir **quão parecida** cada observação é com alguns pontos de referência, chamados **marcos** (*landmarks*).

Uma medida comum de similaridade é a **função de base radial gaussiana** (RBF):

$$\phi_\gamma(\mathbf{x}, \boldsymbol{\ell}) = \exp\left(-\gamma\,\lVert\mathbf{x} - \boldsymbol{\ell}\rVert^2\right)$$

Em palavras: ela vale **1** quando a observação $\mathbf{x}$ está exatamente em cima do marco $\boldsymbol{\ell}$ e cai até perto de **0** à medida que $\mathbf{x}$ se afasta. O formato é de um **sino**, e o $\gamma$ (*gamma*) controla a largura do sino.

Voltamos ao exemplo de uma variável da seção 2.1. Colocamos dois marcos, em $x_1 = -2$ e $x_1 = 1$ (círculos vermelhos), e trocamos $x_1$ por duas variáveis novas:
- $x_2$ = similaridade com o marco em −2;
- $x_3$ = similaridade com o marco em 1.

Por exemplo, para o ponto $x_1 = -1$ (indicado pela seta), $x_2 \approx 0{,}74$ e $x_3 \approx 0{,}30$.

In [ ]:
# Código do gráfico: variáveis de similaridade com dois marcos

def rbf_gaussiana(x, marco, gamma):
    return np.exp(-gamma * np.linalg.norm(x - marco, axis=1) ** 2)

gamma = 0.3

x1s = np.linspace(-4.5, 4.5, 200).reshape(-1, 1)
x2s = rbf_gaussiana(x1s, -2, gamma)
x3s = rbf_gaussiana(x1s, 1, gamma)

XK = np.c_[rbf_gaussiana(X1D, -2, gamma), rbf_gaussiana(X1D, 1, gamma)]
print("Para x1 = -1: x2 =", XK[3, 0].round(2), "| x3 =", XK[3, 1].round(2))

plt.figure(figsize=(10.5, 4))

plt.subplot(121)
plt.grid(alpha=0.25)
plt.axhline(y=0, color="k")
plt.scatter(x=[-2, 1], y=[0, 0], s=150, alpha=0.5, c="red")
plt.plot(X1D[:, 0][y1d == 0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][y1d == 1], np.zeros(5), "g^")
plt.plot(x1s, x2s, "g--")
plt.plot(x1s, x3s, "b:")
plt.gca().get_yaxis().set_ticks([0, 0.25, 0.5, 0.75, 1])
plt.xlabel("$x_1$")
plt.ylabel("Similaridade")
plt.annotate(r"$\mathbf{x}$", xy=(X1D[3, 0], 0), xytext=(-0.5, 0.20), ha="center",
             arrowprops=dict(facecolor="black", shrink=0.1), fontsize=16)
plt.text(-2, 0.9, "$x_2$", ha="center", fontsize=15)
plt.text(1, 0.9, "$x_3$", ha="center", fontsize=15)
plt.axis([-4.5, 4.5, -0.1, 1.1])

plt.subplot(122)
plt.grid(alpha=0.25)
plt.axhline(y=0, color="k")
plt.axvline(x=0, color="k")
plt.plot(XK[:, 0][y1d == 0], XK[:, 1][y1d == 0], "bs")
plt.plot(XK[:, 0][y1d == 1], XK[:, 1][y1d == 1], "g^")
plt.xlabel("$x_2$")
plt.ylabel("$x_3$  ", rotation=0)
plt.annotate(r"$\phi\left(\mathbf{x}\right)$", xy=(XK[3, 0], XK[3, 1]), xytext=(0.65, 0.50),
             ha="center", arrowprops=dict(facecolor="black", shrink=0.1), fontsize=16)
plt.plot([-0.1, 1.1], [0.57, -0.1], "r--", linewidth=3)
plt.axis([-0.1, 1.1, -0.1, 1.1])

plt.tight_layout()
plt.show()

**O que observar:** à esquerda, as curvas em forma de sino mostram a similaridade com cada marco. À direita, cada ponto é representado pelas suas duas similaridades, $(x_2, x_3)$. Nesse novo espaço, **uma reta separa as classes**.

**Como escolher os marcos?** O mais simples é colocar **um marco em cada observação** do treino. Com $m$ observações, ficamos com $m$ variáveis novas: ótimo para separar as classes, mas muito pesado com bases grandes. De novo, o truque do *kernel* resolve isso.

### 2.5 *Kernel* RBF gaussiano

O `SVC(kernel="rbf")` dá o mesmo resultado que criar uma variável de similaridade para **cada observação**, sem criá-las de fato. É o *kernel* **padrão** do `SVC` e o mais usado na prática.

In [ ]:
rbf_kernel_svm_clf = make_pipeline(StandardScaler(),
                                   SVC(kernel="rbf", gamma=5, C=0.001))
rbf_kernel_svm_clf.fit(X, y)

Dois hiperparâmetros controlam o modelo:

- **`gamma`**: a largura do "sino" de similaridade. **Aumentar `gamma`** deixa o sino mais **estreito**: cada observação influencia só a sua vizinhança próxima e a fronteira fica mais **irregular**, contornando pontos individuais. **Diminuir `gamma`** deixa o sino mais largo e a fronteira mais **suave**.
- **`C`**: o mesmo de antes. `C` pequeno → mais regularização.

Ou seja, `gamma` também funciona como um hiperparâmetro de regularização. Vamos comparar quatro combinações:

In [ ]:
# Código do gráfico: efeito de gamma e C no kernel RBF

gamma1, gamma2 = 0.1, 5
C1, C2 = 0.001, 1000
hiperparametros = (gamma1, C1), (gamma1, C2), (gamma2, C1), (gamma2, C2)

svm_clfs = []
for gamma, C in hiperparametros:
    modelo = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma=gamma, C=C))
    modelo.fit(X, y)
    svm_clfs.append(modelo)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10.5, 7), sharex=True, sharey=True)

for i, modelo in enumerate(svm_clfs):
    plt.sca(axes[i // 2, i % 2])
    plotar_previsoes(modelo, [-1.5, 2.45, -1, 1.5])
    plotar_dados(X, y, [-1.5, 2.45, -1, 1.5])
    gamma, C = hiperparametros[i]
    plt.title(f"gamma={gamma}, C={C}")
    if i in (0, 1):
        plt.xlabel("")
    if i in (1, 3):
        plt.ylabel("")

plt.show()

**Interprete**

<details><summary><b>▶ Pergunta 1:</b> compare a linha de cima (<code>gamma=0.1</code>) com a de baixo (<code>gamma=5</code>). O que muda?</summary>
<br>
Com <code>gamma=5</code> (embaixo), a fronteira fica muito mais <b>irregular</b>, com "bolhas" em volta de grupos de pontos. Com <code>gamma=0.1</code> (em cima), ela é <b>suave</b>, quase uma reta curvada. Isso é o efeito do sino estreito (gamma alto) ou largo (gamma baixo).
</details>

<details><summary><b>▶ Pergunta 2:</b> qual painel parece ter <i>underfitting</i>? E qual parece ter <i>overfitting</i>?</summary>
<br>
<b><i>Underfitting</i></b>: em cima à esquerda (<code>gamma=0.1, C=0.001</code>). Regularização demais: a fronteira quase não se curva e erra muitos pontos.<br>
<b><i>Overfitting</i></b>: embaixo à direita (<code>gamma=5, C=1000</code>). Regularização de menos: a fronteira se dobra para acertar quase cada ponto do treino.<br>
Os painéis intermediários ficam mais equilibrados.
</details>

<details><summary><b>▶ Pergunta 3:</b> então como escolher <code>gamma</code> e <code>C</code> na prática?</summary>
<br>
Com <b>busca de hiperparâmetros e validação cruzada</b> (<code>GridSearchCV</code> ou <code>RandomizedSearchCV</code>, Monitoria 2). Como os bons valores podem estar em escalas muito diferentes (0,001 ou 100), é comum testar valores espaçados em escala logarítmica. O exercício 10 (seção 6) faz exatamente isso.<br><br>
Regra de bolso: se houver <i>overfitting</i>, <b>reduza</b> <code>gamma</code> e/ou <code>C</code>; se houver <i>underfitting</i>, <b>aumente</b>.
</details>

### 2.6 Qual classe usar?

Existem outros *kernels* (por exemplo, para textos ou sequências de DNA), mas na prática o caminho costuma ser:

1. **Comece pelo linear** (`LinearSVC`), sobretudo se a base for grande ou tiver muitas variáveis.
2. Se a base for **pequena ou média**, teste também o `SVC` com *kernel* **RBF**, ajustando `gamma` e `C`.
3. Só experimente outros *kernels* se tiver tempo ou um motivo específico.

A tabela compara as classes do scikit-learn para classificação com SVM ($m$ = número de observações, $n$ = número de variáveis):

| Classe | Tempo de treino | Dados que não cabem na memória | Precisa padronizar | Truque do *kernel* |
|---|---|---|---|---|
| `LinearSVC` | $O(m \times n)$ | Não | Sim | Não |
| `SVC` | $O(m^2 \times n)$ a $O(m^3 \times n)$ | Não | Sim | **Sim** |
| `SGDClassifier` | $O(m \times n)$ | **Sim** | Sim | Não |

O ponto mais importante: o tempo do `SVC` cresce com o **quadrado ou o cubo** do número de observações. Ele é ótimo para bases pequenas e médias, mas fica **muito lento com centenas de milhares de linhas**.

<details><summary><b>▶ Pergunta:</b> e o <code>SGDClassifier</code>? Já não usamos ele antes?</summary>
<br>
Sim, na Monitoria 2. Com <code>loss="hinge"</code> (o padrão), o <code>SGDClassifier</code> treina uma <b>SVM linear</b> usando gradiente descendente estocástico (Monitoria 3). É útil para bases enormes ou que chegam aos poucos (aprendizado <i>online</i>), mas não usa o truque do <i>kernel</i>. A seção 5 mostra isso.
</details>

<details><summary><b>▶ Pergunta:</b> e o <code>SVC</code> com mais de duas classes, como no Iris completo?</summary>
<br>
A SVM é, por natureza, um classificador <b>binário</b>. Para várias classes, o scikit-learn combina vários classificadores binários automaticamente (as estratégias OvR e OvO da Monitoria 2). Você não precisa fazer nada: basta chamar <code>fit</code> com um alvo de várias classes. O exercício 10 mostra isso com 3 classes.
</details>

---
## ☕ Intervalo (10 min)

---

## 3. Regressão com SVM

A SVM também serve para **prever números**. A ideia é **inverter o objetivo**:

- **Na classificação**, queremos a rua mais larga com **o mínimo de pontos dentro** dela.
- **Na regressão**, queremos que **o máximo de pontos caiba dentro** da rua, limitando os que ficam de fora.

A largura da rua é o hiperparâmetro **`epsilon`** ($\epsilon$). Erros **menores que $\epsilon$** (pontos dentro da rua) são simplesmente **ignorados**. Por isso dizemos que o modelo é **$\epsilon$-insensível**: acrescentar mais pontos dentro da rua não muda em nada as previsões.

Primeiro, uma regressão linear com `LinearSVR` em dados sintéticos $y = 4 + 3x_1 + \text{ruído}$ (os mesmos da Monitoria 3):

In [ ]:
from sklearn.svm import LinearSVR

np.random.seed(42)
X = 2 * np.random.rand(50, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(50)

svm_reg = make_pipeline(StandardScaler(),
                        LinearSVR(epsilon=0.5, random_state=42))
svm_reg.fit(X, y)

print("Previsão para x = 1:", svm_reg.predict([[1.0]]).round(2))

Agora comparamos duas larguras de rua, $\epsilon = 0{,}5$ e $\epsilon = 1{,}2$. Os pontos destacados em cinza são os **vetores de suporte**: aqui, os que ficam **fora** da rua (ou na borda). São só eles que influenciam o modelo.

In [ ]:
# Código do gráfico: efeito de epsilon na regressão com SVM

def indices_fora_da_rua(modelo, X, y):
    epsilon = modelo[-1].epsilon
    return np.abs(y - modelo.predict(X)) >= epsilon

def plotar_regressao_svm(modelo, X, y, eixos):
    x1s = np.linspace(eixos[0], eixos[1], 100).reshape(100, 1)
    y_pred = modelo.predict(x1s)
    epsilon = modelo[-1].epsilon
    fora = indices_fora_da_rua(modelo, X, y)
    plt.plot(x1s, y_pred, "k-", linewidth=2, label=r"$\hat{y}$", zorder=-2)
    plt.plot(x1s, y_pred + epsilon, "k--", zorder=-2)
    plt.plot(x1s, y_pred - epsilon, "k--", zorder=-2)
    plt.scatter(X[fora], y[fora], s=180, facecolors="#AAA", zorder=-1)
    plt.plot(X, y, "bo")
    plt.xlabel("$x_1$")
    plt.legend(loc="upper left")
    plt.axis(eixos)
    plt.grid(alpha=0.25)

svm_reg2 = make_pipeline(StandardScaler(),
                         LinearSVR(epsilon=1.2, random_state=42))
svm_reg2.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plotar_regressao_svm(svm_reg, X, y, [0, 2, 3, 11])
plt.title(f"epsilon={svm_reg[-1].epsilon}")
plt.ylabel("$y$", rotation=0)

plt.sca(axes[1])
plotar_regressao_svm(svm_reg2, X, y, [0, 2, 3, 11])
plt.title(f"epsilon={svm_reg2[-1].epsilon}")
x_seta = 1
y_seta = svm_reg2.predict([[x_seta]])[0]
plt.annotate("", xy=(x_seta, y_seta), xytext=(x_seta, y_seta - svm_reg2[-1].epsilon),
             arrowprops={"arrowstyle": "<->", "linewidth": 1.5})
plt.text(0.90, 5.4, r"$\epsilon$", fontsize=16)

plt.show()

for modelo in (svm_reg, svm_reg2):
    print(f"epsilon = {modelo[-1].epsilon}: {indices_fora_da_rua(modelo, X, y).sum()} "
          "pontos fora da rua (vetores de suporte)")

**O que observar:** com $\epsilon$ maior (direita), a rua é mais larga, cabem mais pontos dentro dela e **menos pontos** viram vetores de suporte. Reduzir $\epsilon$ aumenta o número de vetores de suporte, o que funciona como **menos regularização**.

Para relações **não lineares**, usamos de novo um *kernel*. O `SVR` é o equivalente do `SVC` para regressão. Testamos em dados quadráticos com *kernel* polinomial de grau 2:

In [ ]:
from sklearn.svm import SVR

np.random.seed(42)
X = 2 * np.random.rand(50, 1) - 1
y = 0.2 + 0.1 * X[:, 0] + 0.5 * X[:, 0] ** 2 + np.random.randn(50) / 10  # relação quadrática

svm_poly_reg = make_pipeline(StandardScaler(),
                             SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1))
svm_poly_reg.fit(X, y)

In [ ]:
# Código do gráfico: SVR com kernel polinomial, C pequeno x C grande

svm_poly_reg2 = make_pipeline(StandardScaler(),
                              SVR(kernel="poly", degree=2, C=100, epsilon=0.1))
svm_poly_reg2.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
for ax, modelo in zip(axes, (svm_poly_reg, svm_poly_reg2)):
    plt.sca(ax)
    plotar_regressao_svm(modelo, X, y, [-1, 1, 0, 1])
    svr = modelo[-1]
    plt.title(f"degree={svr.degree}, C={svr.C}, epsilon={svr.epsilon}", fontsize=12)
plt.sca(axes[0])
plt.ylabel("$y$", rotation=0)
plt.show()

**Interprete**

<details><summary><b>▶ Pergunta 1:</b> qual dos dois modelos está mais regularizado? Como dá para ver no gráfico?</summary>
<br>
O da esquerda (<code>C=0.01</code>). Com <code>C</code> pequeno, a regularização é mais forte e a curva fica mais <b>achatada</b>: o fundo do U fica mais alto e as pontas, mais baixas. Com <code>C=100</code> (direita), a curva se curva mais e acompanha melhor o formato em U dos dados. Aqui a diferença é pequena porque o <i>kernel</i> de grau 2 já tem o formato certo para estes dados. Com um <code>C</code> ainda menor, a curva ficaria cada vez mais reta (<i>underfitting</i>).
</details>

<details><summary><b>▶ Pergunta 2:</b> quando usar SVR em vez de uma regressão linear comum?</summary>
<br>
O SVR é interessante quando a relação é <b>não linear</b> (com <i>kernel</i>) e a base é <b>pequena ou média</b>. Ele também ignora erros pequenos (menores que <code>epsilon</code>), o que pode ser útil quando pequenas diferenças não importam para a decisão.<br><br>
Por outro lado, ele fica <b>lento em bases grandes</b> e nem sempre vence: no exercício 11 (seção 6), em dados de imóveis, ele melhora o modelo linear, mas a Monitoria 2 já mostrou que florestas aleatórias costumam ir melhor nesse tipo de dado. Como sempre, compare por validação cruzada.
</details>

## 4. Por dentro da SVM (*opcional*)

Esta seção mostra **como** a SVM encontra a rua mais larga. É útil para entender os hiperparâmetros, mas não é necessária para usar o modelo.

### 4.1 A função de decisão

Como os modelos lineares da Monitoria 3, a SVM linear calcula uma **pontuação** para cada observação:

$$s = \mathbf{w}^\top\mathbf{x} + b = w_1 x_1 + \dots + w_n x_n + b$$

- Se $s \geq 0$, prevê a classe **positiva**; se $s < 0$, a **negativa**. A fronteira é onde $s = 0$.
- As **bordas da rua** são onde $s = +1$ e $s = -1$.

### 4.2 Rua larga = pesos pequenos

O gráfico mostra o caso de uma variável. A pontuação $s = w_1 x_1$ é uma reta. A rua (faixa cinza) vai do ponto em que $s = -1$ até o ponto em que $s = +1$. Quando o peso $w_1$ cai pela metade (de 1 para 0,5), a reta fica menos inclinada e **a rua dobra de largura**.

Conclusão: **para alargar a rua, a SVM precisa deixar os pesos $\mathbf{w}$ pequenos.**

In [ ]:
# Código do gráfico: pesos menores dão uma rua mais larga

import matplotlib.patches as patches

def plotar_funcao_decisao_1d(w, b, mostrar_rotulos=True, x1_lim=(-3, 3)):
    x1 = np.linspace(x1_lim[0], x1_lim[1], 200)
    s = w * x1 + b
    meia_margem = 1 / w

    plt.plot(x1, s, "b-", linewidth=2, label=r"$s = w_1 x_1$")
    plt.axhline(y=0, color="k", linewidth=1)
    plt.axvline(x=0, color="k", linewidth=1)
    rua = patches.Rectangle((-meia_margem, -2), 2 * meia_margem, 4,
                            edgecolor="none", facecolor="gray", alpha=0.2)
    plt.gca().add_patch(rua)
    plt.plot([-3, 3], [1, 1], "k--", linewidth=1)
    plt.plot([-3, 3], [-1, -1], "k--", linewidth=1)
    plt.plot(meia_margem, 1, "k.")
    plt.plot(-meia_margem, -1, "k.")
    plt.axis(list(x1_lim) + [-2, 2])
    plt.xlabel("$x_1$")
    if mostrar_rotulos:
        plt.ylabel("$s$", rotation=0, labelpad=5)
        plt.legend()
        plt.text(1.02, -1.6, "Margem", ha="left", va="center", color="k")
    plt.annotate("", xy=(-meia_margem, -1.6), xytext=(meia_margem, -1.6),
                 arrowprops={"ec": "k", "arrowstyle": "<->", "linewidth": 1.5})
    plt.title(f"$w_1 = {w}$")
    plt.grid(alpha=0.25)

fig, axes = plt.subplots(ncols=2, figsize=(9, 3.2), sharey=True)
plt.sca(axes[0])
plotar_funcao_decisao_1d(1, 0)
plt.sca(axes[1])
plotar_funcao_decisao_1d(0.5, 0, mostrar_rotulos=False)
plt.show()

### 4.3 O problema de otimização

Codificamos as classes como $t = +1$ (positiva) e $t = -1$ (negativa).

**Margem rígida:** queremos pesos pequenos com **todas** as observações fora da rua e do lado certo:

$$\min_{\mathbf{w}, b}\ \frac{1}{2}\mathbf{w}^\top\mathbf{w} \quad \text{sujeito a} \quad t^{(i)}\left(\mathbf{w}^\top\mathbf{x}^{(i)} + b\right) \geq 1 \ \text{ para toda observação } i$$

(Minimizamos $\frac{1}{2}\lVert\mathbf{w}\rVert^2$ em vez de $\lVert\mathbf{w}\rVert$ só porque é mais fácil de derivar.)

**Margem suave:** permitimos violações, mas cada uma tem um custo. Isso equivale a minimizar:

$$J(\mathbf{w}, b) = \underbrace{\frac{1}{2}\mathbf{w}^\top\mathbf{w}}_{\text{rua larga}} \; + \; C\sum_{i=1}^{m}\underbrace{\max\left(0,\ 1 - t^{(i)} s^{(i)}\right)}_{\textit{hinge loss}}$$

Compare com a Ridge da Monitoria 3 ($\text{MSE} + \alpha \cdot \text{penalidade}$). A estrutura é a mesma: **erro + penalidade nos pesos**. A diferença é que aqui o `C` multiplica o **erro**, e não a penalidade. Por isso `C` grande significa **menos** regularização.

### 4.4 A *hinge loss*

A **hinge loss** ("perda dobradiça") mede o custo de cada observação:

- **zero** se a observação está do lado certo e **fora da rua** ($t \cdot s \geq 1$);
- cresce **linearmente** à medida que ela entra na rua ou passa para o lado errado.

O `LinearSVC` usa, por padrão, a **hinge loss ao quadrado** (`loss="squared_hinge"`), que pune mais fortemente os pontos muito errados.

In [ ]:
# Código do gráfico: hinge loss e hinge loss ao quadrado

s = np.linspace(-2.5, 2.5, 200)
hinge_pos = np.where(1 - s < 0, 0, 1 - s)  # max(0, 1 - s), para t = +1
hinge_neg = np.where(1 + s < 0, 0, 1 + s)  # max(0, 1 + s), para t = -1

titulos = (r"Hinge loss = $\max(0, 1 - s\,t)$", "Hinge loss ao quadrado")

fig, axs = plt.subplots(1, 2, sharey=True, figsize=(8.2, 3))

for ax, perda_pos, perda_neg, titulo in zip(
        axs, (hinge_pos, hinge_pos ** 2), (hinge_neg, hinge_neg ** 2), titulos):
    ax.plot(s, perda_pos, "g-", linewidth=2, zorder=10, label="$t=1$")
    ax.plot(s, perda_neg, "r--", linewidth=2, zorder=10, label="$t=-1$")
    ax.grid(alpha=0.25)
    ax.axhline(y=0, color="k")
    ax.axvline(x=0, color="k")
    ax.set_xlabel(r"$s = \mathbf{w}^\top \mathbf{x} + b$")
    ax.axis([-2.5, 2.5, -0.5, 2.5])
    ax.legend(loc="center right")
    ax.set_title(titulo, fontsize=12)
    ax.set_yticks(np.arange(0, 2.5, 1))
    ax.set_aspect("equal")

plt.show()

**O que observar:** a linha verde é a perda de uma observação positiva ($t = 1$). Ela é zero para $s \geq 1$ (lado certo, fora da rua) e cresce à medida que $s$ diminui. A vermelha é o espelho para uma observação negativa.

<details><summary><b>▶ Pergunta:</b> o que é o "problema dual" e por que ele importa para o <i>kernel</i>?</summary>
<br>
O problema de otimização acima (o "primal") pode ser reescrito numa forma equivalente, chamada <b>dual</b>. Na forma dual, os dados aparecem <b>apenas</b> em produtos escalares entre pares de observações, do tipo <code>x(i) · x(j)</code>.<br><br>
É isso que permite o truque do <i>kernel</i>: basta trocar cada produto escalar por uma função <i>kernel</i> <code>K(x(i), x(j))</code>, que mede a semelhança como se os dados estivessem num espaço com muito mais variáveis.<br><br>
O dual também revela por que só os vetores de suporte importam: na solução, todas as outras observações recebem peso zero. O parâmetro <code>dual</code> do <code>LinearSVC</code> escolhe qual das duas formas resolver (o padrão, <code>"auto"</code>, escolhe sozinho).
</details>

## 5. SVM "do zero" com gradiente descendente (*opcional*)

Como a função de custo da seção 4.3 é **convexa**, podemos minimizá-la com o **gradiente descendente em lote** da Monitoria 3. Fazer isso à mão ajuda a ver que não há mágica dentro do `LinearSVC`.

Voltamos ao Iris: *virginica* ou não, com comprimento e largura da pétala.

In [ ]:
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 2).values  # True se for virginica

A classe abaixo segue o padrão do scikit-learn (`fit`, `predict`, `decision_function`). Em cada época:

1. identifica as observações que estão **na rua ou do lado errado** (as únicas com *hinge loss* positiva);
2. calcula o gradiente do custo $J$ usando só essas observações;
3. dá um passo na direção oposta ao gradiente, com uma taxa de aprendizado que diminui ao longo do tempo.

In [ ]:
from sklearn.base import BaseEstimator

class MeuLinearSVC(BaseEstimator):
    def __init__(self, C=1, eta0=1, eta_d=10000, n_epocas=1000, random_state=None):
        self.C = C
        self.eta0 = eta0
        self.eta_d = eta_d
        self.n_epocas = n_epocas
        self.random_state = random_state

    def eta(self, epoca):
        return self.eta0 / (epoca + self.eta_d)  # taxa de aprendizado decrescente

    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)
        w = np.random.randn(X.shape[1], 1)  # pesos iniciais aleatórios
        b = 0

        t = np.array(y, dtype=np.float64).reshape(-1, 1) * 2 - 1  # +1 ou -1
        X_t = X * t
        self.Js = []  # guarda o custo de cada época

        for epoca in range(self.n_epocas):
            # 1. observações na rua ou do lado errado
            idx_suporte = (X_t @ w + t * b < 1).ravel()
            X_t_sv = X_t[idx_suporte]
            t_sv = t[idx_suporte]

            # custo J = 1/2 ||w||² + C * soma das hinge losses
            J = 1 / 2 * (w * w).sum() + self.C * ((1 - X_t_sv @ w).sum() - b * t_sv.sum())
            self.Js.append(J)

            # 2. gradientes
            gradiente_w = w - self.C * X_t_sv.sum(axis=0).reshape(-1, 1)
            derivada_b = -self.C * t_sv.sum()

            # 3. passo do gradiente descendente
            w = w - self.eta(epoca) * gradiente_w
            b = b - self.eta(epoca) * derivada_b

        self.intercept_ = np.array([b])
        self.coef_ = w.reshape(1, -1)
        idx_suporte = (X_t @ w + t * b < 1).ravel()
        self.support_vectors_ = X[idx_suporte]
        return self

    def decision_function(self, X):
        return X @ self.coef_[0] + self.intercept_[0]

    def predict(self, X):
        return self.decision_function(X) >= 0

Treinamos com `C = 2` e prevemos duas flores (5 × 2 cm e 4 × 1 cm). As 60 mil épocas levam alguns segundos.

In [ ]:
C = 2
meu_svm = MeuLinearSVC(C=C, eta0=10, eta_d=1000, n_epocas=60_000, random_state=2)
meu_svm.fit(X, y)
print("É virginica? [5 x 2 cm, 4 x 1 cm]:", meu_svm.predict(np.array([[5, 2], [4, 1]])))

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(range(meu_svm.n_epocas), meu_svm.Js)
plt.axis([0, meu_svm.n_epocas, 0, 100])
plt.xlabel("Época")
plt.ylabel("Custo $J$")
plt.grid(alpha=0.25)
plt.show()

O custo cai rapidamente e se estabiliza: o gradiente descendente convergiu. Agora comparamos os parâmetros com os do `SVC(kernel="linear")` do scikit-learn, treinado com o mesmo `C`:

In [ ]:
svc_sklearn = SVC(kernel="linear", C=C)
svc_sklearn.fit(X, y)

print("MeuLinearSVC -> b:", meu_svm.intercept_.round(2), "| w:", meu_svm.coef_.round(2))
print("SVC          -> b:", svc_sklearn.intercept_.round(2), "| w:", svc_sklearn.coef_.round(2))

In [ ]:
# Código do gráfico: nossa implementação x scikit-learn

fig, axes = plt.subplots(ncols=2, figsize=(11, 3.2), sharey=True)
for ax, modelo, titulo in zip(axes, (meu_svm, svc_sklearn), ("MeuLinearSVC", "SVC")):
    plt.sca(ax)
    plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^", label="Iris virginica")
    plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs", label="Não virginica")
    plotar_fronteira_svc(modelo, 4, 6)
    plt.xlabel("Comprimento da pétala")
    plt.title(titulo)
    plt.axis([4, 6, 0.8, 2.8])
    plt.grid(alpha=0.25)

plt.sca(axes[0])
plt.ylabel("Largura da pétala")
plt.legend(loc="upper left", fontsize=11)
plt.show()

As duas fronteiras são praticamente iguais.

Por fim, o `SGDClassifier` com `loss="hinge"` treina o **mesmo tipo de modelo** com gradiente descendente **estocástico**. Ele não tem `C`; a regularização é controlada por `alpha` (maior `alpha` → mais regularização, como na Ridge).

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(loss="hinge", alpha=0.017, max_iter=1000, tol=1e-3,
                        random_state=42)
sgd_clf.fit(X, y)

w_sgd, b_sgd = sgd_clf.coef_[0], sgd_clf.intercept_[0]
print("SGDClassifier -> b:", round(b_sgd, 2), "| w:", w_sgd.round(2))

plt.figure(figsize=(5.5, 3.2))
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^")
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs")
plotar_rua(w_sgd, b_sgd, 4, 6, vetores_de_suporte(X, y, w_sgd, b_sgd))
plt.xlabel("Comprimento da pétala")
plt.ylabel("Largura da pétala")
plt.title("SGDClassifier")
plt.axis([4, 6, 0.8, 2.8])
plt.grid(alpha=0.25)
plt.show()

## 6. Exercícios resolvidos do capítulo

Estes são os exercícios práticos 9, 10 e 11 do capítulo 5, **com as soluções do livro**. Para aproveitar melhor:

1. Leia o enunciado.
2. **Pense (ou escreva) como você resolveria** antes de executar as células de solução.
3. Execute a solução e leia os comentários entre as células.

### Exercício 9: três classes, o mesmo modelo

> **Enunciado:** treine um `LinearSVC` num conjunto de dados linearmente separável. Depois, treine um `SVC` e um `SGDClassifier` no mesmo conjunto. Veja se consegue fazer os três produzirem aproximadamente o mesmo modelo.

**Solução.** Usamos o Iris com *setosa* e *versicolor*, que são linearmente separáveis (seção 1.1).

In [ ]:
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris.target.values

setosa_ou_versicolor = (y == 0) | (y == 1)
X = X[setosa_ou_versicolor]
y = y[setosa_ou_versicolor]

Para os três modelos ficarem parecidos, três ajustes são necessários:

- O `LinearSVC` usa `loss="squared_hinge"` por padrão. Para igualar aos outros, usamos `loss="hinge"` (que exige `dual=True`).
- O `SVC` usa o *kernel* RBF por padrão. Precisamos de `kernel="linear"`.
- O `SGDClassifier` não tem o hiperparâmetro `C`, mas tem `alpha`. Ajustamos `alpha` até a fronteira ficar parecida.

In [ ]:
C = 5
alpha = 0.05

scaler = StandardScaler()
X_padronizado = scaler.fit_transform(X)

lin_clf = LinearSVC(loss="hinge", C=C, dual=True, random_state=42).fit(X_padronizado, y)
svc_clf = SVC(kernel="linear", C=C).fit(X_padronizado, y)
sgd_clf = SGDClassifier(alpha=alpha, random_state=42).fit(X_padronizado, y)

Vamos desenhar as três fronteiras, já convertidas de volta para a escala original (centímetros):

In [ ]:
def fronteira_na_escala_original(modelo):
    # Dois pontos da reta de decisão (no espaço padronizado), convertidos para centímetros
    w = -modelo.coef_[0, 0] / modelo.coef_[0, 1]
    b = -modelo.intercept_[0] / modelo.coef_[0, 1]
    return scaler.inverse_transform([[-10, -10 * w + b], [10, 10 * w + b]])

linha_lin = fronteira_na_escala_original(lin_clf)
linha_svc = fronteira_na_escala_original(svc_clf)
linha_sgd = fronteira_na_escala_original(sgd_clf)

plt.figure(figsize=(11, 4))
plt.plot(linha_lin[:, 0], linha_lin[:, 1], "k:", label="LinearSVC")
plt.plot(linha_svc[:, 0], linha_svc[:, 1], "b--", linewidth=2, label="SVC")
plt.plot(linha_sgd[:, 0], linha_sgd[:, 1], "r-", label="SGDClassifier")
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "bs")  # Iris versicolor
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "yo")  # Iris setosa
plt.xlabel("Comprimento da pétala")
plt.ylabel("Largura da pétala")
plt.legend(loc="upper center")
plt.axis([0, 5.5, 0, 2])
plt.grid(alpha=0.25)
plt.show()

As três fronteiras ficam **muito próximas**. Conclusão: são três algoritmos de treino diferentes para o **mesmo tipo de modelo** (uma SVM linear).

### Exercício 10: vinhos

> **Enunciado:** treine um classificador SVM na base *Wine* (`sklearn.datasets.load_wine()`). Ela tem a análise química de 178 vinhos de **3 produtores** diferentes. O objetivo é prever o produtor a partir da análise química. Que acurácia você consegue?

**Solução.** Carregamos a base, olhamos os dados e separamos treino e teste.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

wine = load_wine(as_frame=True)
print("Formato:", wine.data.shape)
print("Vinhos por produtor:", wine.target.value_counts().sort_index().tolist())

X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, random_state=42)

X_train.head()

São 13 variáveis químicas (álcool, ácido málico, magnésio, ...). Repare que as **escalas são muito diferentes**: `magnesium` está na casa de 100, `hue` perto de 1 e `proline` na casa de 1.000. Guarde isso.

Começamos simples, com uma SVM linear. Ela lida com as 3 classes automaticamente (estratégia "um contra o resto", OvR, da Monitoria 2).

> ⚠️ **Um aviso (`ConvergenceWarning`) vai aparecer. É proposital.** Ele avisa que o algoritmo não conseguiu terminar a otimização.

In [ ]:
lin_clf = LinearSVC(dual=True, random_state=42)
lin_clf.fit(X_train, y_train)

O modelo **não convergiu**. Será que basta aumentar o número de iterações? Vamos tentar com 1 milhão (pode levar alguns segundos):

In [ ]:
lin_clf = LinearSVC(max_iter=1_000_000, dual=True, random_state=42)
lin_clf.fit(X_train, y_train)

Mesmo com um milhão de iterações, ainda não convergiu. O problema é outro. Mesmo assim, vamos medir a acurácia por validação cruzada para ter uma **linha de base**:

In [ ]:
from sklearn.model_selection import cross_val_score

print("Acurácia (validação cruzada):", cross_val_score(lin_clf, X_train, y_train).mean().round(3))

<details><summary><b>▶ Pergunta:</b> antes de continuar, você consegue adivinhar qual é o problema?</summary>
<br>
<b>Esquecemos de padronizar as variáveis!</b> (Lembre da seção 1.2 e das escalas que vimos no <code>head()</code>.) Com escalas tão diferentes, a otimização fica difícil e a SVM praticamente ignora as variáveis de escala pequena.
</details>

Corrigimos colocando o `StandardScaler` no pipeline:

In [ ]:
lin_clf = make_pipeline(StandardScaler(),
                        LinearSVC(dual=True, random_state=42))
lin_clf.fit(X_train, y_train)

print("Acurácia (validação cruzada):", cross_val_score(lin_clf, X_train, y_train).mean().round(3))

Agora converge sem problemas e a acurácia melhora bastante. **Sempre padronize as variáveis com SVM.**

Será que uma SVM com *kernel* (RBF, o padrão do `SVC`) faz melhor? Primeiro com os hiperparâmetros padrão:

In [ ]:
svm_clf = make_pipeline(StandardScaler(), SVC(random_state=42))
print("Acurácia (validação cruzada):", cross_val_score(svm_clf, X_train, y_train).mean().round(3))

Não melhorou. Talvez seja preciso **ajustar os hiperparâmetros**. Usamos o `RandomizedSearchCV` (Monitoria 2), que sorteia 100 combinações de `gamma` e `C` e avalia cada uma por validação cruzada:

- `loguniform(0.001, 0.1)` sorteia `gamma` de forma uniforme **na escala logarítmica**: valores perto de 0,001 são tão prováveis quanto perto de 0,1;
- `uniform(1, 10)` sorteia `C` entre 1 e 11.

O prefixo `svc__` indica que o hiperparâmetro é da etapa `svc` do pipeline. A busca pode levar alguns segundos.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform

param_distrib = {
    "svc__gamma": loguniform(0.001, 0.1),
    "svc__C": uniform(1, 10),
}
rnd_search_cv = RandomizedSearchCV(svm_clf, param_distrib, n_iter=100, cv=5,
                                   random_state=42)
rnd_search_cv.fit(X_train, y_train)

melhores = rnd_search_cv.best_params_
print(f"Melhores hiperparâmetros: gamma = {melhores['svc__gamma']:.4f} | C = {melhores['svc__C']:.2f}")
print("Acurácia (validação cruzada):", round(rnd_search_cv.best_score_, 3))

Excelente. Escolhemos esse modelo e, só agora, olhamos o **conjunto de teste** (uma única vez):

In [ ]:
print("Acurácia no teste:", round(rnd_search_cv.score(X_test, y_test), 3))

A SVM com *kernel* ajustada supera o `LinearSVC`, mas a acurácia no teste ficou **um pouco abaixo** da validação cruzada. Isso é comum: depois de testar 100 combinações, acabamos com um leve *overfitting* aos próprios conjuntos de validação.

É tentador continuar ajustando até melhorar o resultado no teste, mas isso **não** ajudaria: passaríamos a fazer *overfitting* no teste, e ele deixaria de ser uma medida honesta do desempenho em dados novos. O resultado já é muito bom, então paramos aqui.

### Exercício 11: preços de imóveis na Califórnia

> **Enunciado:** treine e ajuste um regressor SVM na base *California housing* (`sklearn.datasets.fetch_california_housing()`). O alvo é o valor mediano dos imóveis de cada região, em **centenas de milhares de dólares**. Como há mais de 20 mil observações, a SVM pode ficar lenta; para ajustar os hiperparâmetros, use bem menos observações (por exemplo, 2.000). Qual o RMSE do seu melhor modelo?

> ⏱️ **Este exercício é o mais lento da aula** (alguns minutos no Colab), justamente porque o `SVR` fica lento com muitos dados (seção 2.6). Também é preciso internet para baixar a base.

**Solução.** Carregamos a base e separamos treino (80%) e teste (20%):

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = housing.data
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

Começamos com um `LinearSVR` simples (sem esquecer de padronizar).

> ⚠️ Outro `ConvergenceWarning` vai aparecer. É esperado.

In [ ]:
lin_svr = make_pipeline(StandardScaler(), LinearSVR(dual=True, random_state=42))
lin_svr.fit(X_train, y_train)

Não convergiu. Aqui, como as variáveis **já estão padronizadas**, faz sentido aumentar `max_iter`:

In [ ]:
lin_svr = make_pipeline(StandardScaler(),
                        LinearSVR(max_iter=5000, dual=True, random_state=42))
lin_svr.fit(X_train, y_train)

Agora medimos o erro no treino com o **RMSE** (raiz do erro quadrático médio, Monitoria 3).

In [ ]:
from sklearn.metrics import root_mean_squared_error

y_pred = lin_svr.predict(X_train)
rmse = root_mean_squared_error(y_train, y_pred)
print(f"RMSE no treino: {rmse:.3f}  (erro típico de cerca de US$ {rmse * 100:.0f} mil)")

Como o alvo está em centenas de milhares de dólares, esse RMSE indica erros típicos perto de **US\$ 98 mil**. Não é bom. Vamos tentar um `SVR` com *kernel* RBF, buscando `C` e `gamma` com `RandomizedSearchCV`.

Para a busca ser rápida, usamos só as **primeiras 2.000** observações do treino (seção 2.6: o `SVC`/`SVR` fica lento com muitas linhas).

In [ ]:
svm_reg = make_pipeline(StandardScaler(), SVR())

param_distrib = {
    "svr__gamma": loguniform(0.001, 0.1),
    "svr__C": uniform(1, 10),
}
rnd_search_cv = RandomizedSearchCV(svm_reg, param_distrib,
                                   n_iter=100, cv=3, random_state=42)
rnd_search_cv.fit(X_train[:2000], y_train[:2000])

melhores = rnd_search_cv.best_params_
print(f"Melhores hiperparâmetros: gamma = {melhores['svr__gamma']:.4f} | C = {melhores['svr__C']:.2f}")

Agora avaliamos o melhor modelo por validação cruzada no **treino completo**. Esta é a célula mais demorada (alguns minutos): são 5 treinos de `SVR`, cada um com mais de 13 mil observações.

O scikit-learn devolve o RMSE com sinal negativo (porque ele sempre **maximiza** a pontuação), então colocamos um `-` na frente.

In [ ]:
rmse_cv = -cross_val_score(rnd_search_cv.best_estimator_, X_train, y_train,
                           scoring="neg_root_mean_squared_error")
print("RMSE em cada parte da validação cruzada:", rmse_cv.round(3))
print("RMSE médio:", rmse_cv.mean().round(3))

Bem melhor que o modelo linear. Escolhemos esse modelo e o avaliamos no **teste**:

In [ ]:
y_pred = rnd_search_cv.best_estimator_.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"RMSE no teste: {rmse:.3f}  (erro típico de cerca de US$ {rmse * 100:.0f} mil)")

**Conclusão:** a SVM funcionou muito bem na base de vinhos, mas nem tanto na de imóveis. No capítulo 2 do livro (e na Monitoria 2), florestas aleatórias se saíram melhor nesse tipo de dado. **Não existe modelo que vence sempre**: compare as alternativas por validação cruzada.

## 7. Usar IA nesta aula

Como você está estudando sozinho, a IA é uma boa "monitora de plantão". Alguns usos:

**Para aprender:** peça uma explicação com analogia.
> Explique o que é uma SVM (máquina de vetores de suporte) para alguém sem formação em estatística, usando a analogia de construir a rua mais larga possível entre dois bairros. Explique também o que são os vetores de suporte e o que o hiperparâmetro C controla.

**Para entender um gráfico:** descreva o que você vê.
> Treinei uma SVM com kernel RBF e, com gamma=5 e C=1000, a fronteira de decisão ficou cheia de "bolhas" em volta dos pontos de treino. O que isso indica? O que devo fazer com gamma e C, e como verifico se melhorou?

**Para decidir:** descreva o problema de negócio.
> Tenho uma base com 500 mil clientes e 40 variáveis e quero prever se cada cliente vai cancelar o contrato. Vale a pena usar SVC com kernel RBF? Compare com LinearSVC, SGDClassifier e regressão logística, considerando tempo de treino e a necessidade de probabilidades para calcular o valor esperado de uma ação de retenção.

**Para depurar:** cole o erro completo, a célula e o que você esperava que acontecesse (o mesmo roteiro da Monitoria 1). Por exemplo, peça para a IA explicar o `ConvergenceWarning` do exercício 10.

Lembre-se: **confira** o que a IA responde, executando o código e comparando com o que você viu nesta aula.

## 8. Fechamento

Ao terminar, você deve conseguir:
- Explicar a ideia de **classificação com margem larga** e o papel dos **vetores de suporte**.
- Explicar por que é preciso **padronizar** as variáveis antes de usar SVM.
- Diferenciar **margem rígida** e **margem suave** e interpretar o hiperparâmetro **`C`**.
- Usar variáveis polinomiais e o **truque do *kernel*** (polinomial e RBF) para fronteiras não lineares.
- Interpretar **`gamma`** e **`C`** e reconhecer *underfitting* e *overfitting* nos gráficos.
- Escolher entre `LinearSVC`, `SVC` e `SGDClassifier` conforme o tamanho da base.
- Usar SVM para **regressão** (`LinearSVR` e `SVR`) e interpretar **`epsilon`**.

### Teste seus conhecimentos

Estas são as perguntas conceituais do capítulo (1 a 8). Tente responder antes de abrir cada resposta.

1. Qual é a ideia fundamental por trás das SVM?
<details><summary>Resposta</summary>
Ajustar a <b>"rua" mais larga possível</b> entre as classes, ou seja, ter a maior margem possível entre a fronteira de decisão e as observações de treino. Na margem suave, a SVM busca um meio-termo entre separar perfeitamente as classes e ter a rua mais larga (algumas observações podem ficar dentro da rua). Outra ideia-chave é usar <i>kernels</i> em dados não lineares. A SVM também pode fazer regressão linear e não linear e detecção de novidades.
</details>

2. O que é um vetor de suporte?
<details><summary>Resposta</summary>
Depois do treino, é qualquer observação que fica <b>na rua</b>, incluindo a borda. A fronteira de decisão é inteiramente determinada pelos vetores de suporte. Observações fora da rua não têm influência nenhuma: podemos removê-las, adicionar outras ou movê-las, e, enquanto ficarem fora da rua, a fronteira não muda. As previsões de uma SVM com <i>kernel</i> também só usam os vetores de suporte, não a base de treino inteira.
</details>

3. Por que é importante padronizar as variáveis ao usar SVM?
<details><summary>Resposta</summary>
Porque a SVM procura a rua mais larga, e a largura é medida em distância. Sem padronização, a SVM tende a <b>ignorar as variáveis de escala pequena</b> (seção 1.2).
</details>

4. Uma SVM consegue dar uma pontuação de confiança ao classificar uma observação? E uma probabilidade?
<details><summary>Resposta</summary>
Sim, com o método <code>decision_function()</code>. A pontuação representa a distância (com sinal) até a fronteira de decisão, mas <b>não</b> pode ser lida diretamente como probabilidade. Com <code>SVC(probability=True)</code>, ao final do treino o modelo usa validação cruzada (5 partes) para gerar pontuações fora da amostra e ajusta uma regressão logística que as converte em probabilidades. Aí os métodos <code>predict_proba()</code> e <code>predict_log_proba()</code> ficam disponíveis.
</details>

5. Como escolher entre <code>LinearSVC</code>, <code>SVC</code> e <code>SGDClassifier</code>?
<details><summary>Resposta</summary>
Os três fazem classificação linear com margem larga. O <code>SVC</code> também suporta o truque do <i>kernel</i>, o que permite resolver problemas não lineares, mas tem um custo: <b>não escala bem com muitas observações</b> (escala bem com muitas variáveis). O <code>LinearSVC</code> é um algoritmo otimizado para SVM linear. O <code>SGDClassifier</code> usa gradiente descendente estocástico; dependendo da base, o <code>LinearSVC</code> pode ser um pouco mais rápido, mas o <code>SGDClassifier</code> é mais flexível e permite aprendizado incremental (dados que chegam aos poucos ou não cabem na memória).
</details>

6. Você treinou uma SVM com <i>kernel</i> RBF e ela tem <i>underfitting</i> no treino. O que fazer com <code>gamma</code> e <code>C</code>?
<details><summary>Resposta</summary>
Provavelmente há regularização demais. Para reduzi-la, <b>aumente <code>gamma</code> ou <code>C</code></b> (ou os dois).
</details>

7. O que significa dizer que um modelo SVM de regressão é <i>ϵ-insensível</i>?
<details><summary>Resposta</summary>
A SVM de regressão tenta fazer caber o máximo possível de observações dentro de uma rua estreita em torno das previsões. Se você adicionar observações <b>dentro</b> dessa rua (de largura controlada por <code>epsilon</code>), o modelo não muda em nada.
</details>

8. Qual é o objetivo do truque do <i>kernel</i>?
<details><summary>Resposta</summary>
É uma técnica matemática que permite treinar uma SVM não linear. O resultado equivale a transformar as variáveis num espaço com muito mais dimensões (por uma transformação não linear) e treinar uma SVM linear nesse espaço, <b>sem precisar fazer a transformação</b>.
</details>

### Antes de sair
1. Um conceito que ficou claro.
2. Uma dúvida que ainda tem (anote para levar à próxima aula).
3. Um problema do seu trabalho em que uma SVM poderia ser útil, e se você usaria o *kernel* linear ou RBF.

### Referências
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow* (3ª ed.), capítulo 5. [Notebook](https://github.com/ageron/handson-ml3/blob/main/05_support_vector_machines.ipynb): `05_support_vector_machines.ipynb`.
- Guia do scikit-learn: [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html).